In [ ]:
# [Cell 1] ติดตั้งโปรแกรมและสตาร์ทเซิร์ฟเวอร์พื้นฐาน
# 1. ติดตั้งแพ็กเกจระดับ OS
!sudo apt-get update && sudo apt-get install -y zstd tesseract-ocr tesseract-ocr-tha poppler-utils
!curl -fsSL https://ollama.com/install.sh | sh

# 2. ติดตั้ง Python Library (เพิ่ม jedi เพื่อแก้ warning ของ ipython)
!pip install -q fastapi uvicorn python-multipart pyngrok nest-asyncio pypdf pdf2image pytesseract jedi
!pip install -q llama-index llama-index-llms-ollama llama-index-embeddings-huggingface llama-index-vector-stores-qdrant llama-index-readers-file qdrant-client

import subprocess
import time

# 3. เคลียร์โปรเซส Ollama และเปิดใหม่
!pkill ollama
time.sleep(2)
print("🚀 กำลังสตาร์ท Ollama Server...")
subprocess.Popen(["ollama", "serve"])
time.sleep(5)

# 4. โหลดโมเดล (แก้ไขชื่อโมเดลให้ถูกต้องตาม Ollama Library)
# ใช้เป็น llama3 หรือตัวอื่นที่ชัวร์ว่ามีอยู่ก่อนเพื่อทดสอบระบบ
print("📥 กำลังดาวน์โหลดโมเดล...")
!ollama pull scb10x/typhoon2.5-qwen3-4b
!ollama pull iapp/chinda-qwen3-4b
print("✅ ติดตั้งและโหลดโมเดลเสร็จสิ้น!")

In [7]:
# [Cell 2] ตั้งค่า Database และ Embedding (ไม่โหลด LLM ค้างไว้แล้ว)
from llama_index.core import VectorStoreIndex, Settings, StorageContext
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.qdrant import QdrantVectorStore
import qdrant_client
import os

print("🧠 กำลังตั้งค่า Embedding และ Database...")

# 1. โหลดเฉพาะ Embedding Model
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-m3")
Settings.embed_model.get_text_embedding("ทดสอบ")

# 2. เตรียมฐานข้อมูล Qdrant
try:
    # พยายามปิด client เก่าถ้ามี
    if 'client' in globals():
        client.close()
        print("🔒 ปิดการเชื่อมต่อเดิมเรียบร้อย")
except:
    pass

client = qdrant_client.QdrantClient(path="./qdrant_local_data")
vector_store = QdrantVectorStore(client=client, collection_name="tor_documents")
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# ปิด client ทันทีหลังจากตั้งค่าเสร็จเพื่อไม่ให้ lock ไฟล์ทิ้งไว้
client.close()

print("✅ ตั้งค่า Database พร้อมใช้งาน! (LLM จะถูกโหลดเมื่อมีการเรียกใช้ API เท่านั้น)")

🧠 กำลังตั้งค่า Embedding และ Database...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

🔒 ปิดการเชื่อมต่อเดิมเรียบร้อย
✅ ตั้งค่า Database พร้อมใช้งาน! (LLM จะถูกโหลดเมื่อมีการเรียกใช้ API เท่านั้น)


In [9]:
# [Cell 3] ระบบ API แบบแยก Session (Multi-Chat Isolation) พร้อม Debug Log
import os
import shutil
import asyncio
import pypdf
import pytesseract
import json
import re
import subprocess
import time
from pdf2image import convert_from_path
from fastapi import FastAPI, UploadFile, File, BackgroundTasks, Form
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import List
from llama_index.core import Document, VectorStoreIndex, StorageContext
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.llms.ollama import Ollama
import qdrant_client
import nest_asyncio
from pyngrok import ngrok
import uvicorn

# รีสตาร์ท Ollama เพื่อความสดใหม่
!pkill ollama
time.sleep(2)
print("🚀 กำลังสตาร์ท Ollama Server...")
subprocess.Popen(["ollama", "serve"])
time.sleep(5)

app = FastAPI(title="NotebookLM Clone API (Multi-Session)")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

doc_status = {}

# เคลียร์ Lock File ของ Qdrant (ถ้ามี)
if os.path.exists("./qdrant_local_data/.lock"):
    try:
        os.remove("./qdrant_local_data/.lock")
        print("🔓 เคลียร์ Lock File สำเร็จ")
    except:
        print("⚠️ ไม่สามารถลบ Lock File ได้โดยตรง")

# ใช้ Global Client เพื่อเลี่ยงปัญหา Lock File
print("🔗 กำลังเชื่อมต่อ Database...")
global_qdrant_client = qdrant_client.QdrantClient(path="./qdrant_local_data")

def get_session_storage(session_id: str):
    collection_name = f"chat_{session_id.lower().replace('-', '_')}"
    vector_store = QdrantVectorStore(client=global_qdrant_client, collection_name=collection_name)
    return StorageContext.from_defaults(vector_store=vector_store)

def extract_text_smart(file_path):
    text = ""
    try:
        reader = pypdf.PdfReader(file_path)
        for page in reader.pages:
            extracted = page.extract_text()
            if extracted: text += extracted + "\n"
    except: pass
    if len(text.strip()) < 100:
        try:
            print(f"🔍 [OCR] กำลังใช้ Tesseract อ่านไฟล์ภาพ: {os.path.basename(file_path)}")
            images = convert_from_path(file_path, dpi=300)
            for img in images: text += pytesseract.image_to_string(img, lang='tha+eng') + "\n"
        except: pass
    return text

async def process_document_background(file_path: str, filename: str, session_id: str):
    task_id = f"{session_id}_{filename}"
    try:
        print(f"\n[Task] เริ่มสกัดข้อความไฟล์: {filename}")
        extracted_text = extract_text_smart(file_path)
        doc = Document(text=extracted_text, metadata={"file_name": filename})

        print(f"[Task] กำลังสร้าง Vector Index...")
        storage_context = get_session_storage(session_id)
        index = VectorStoreIndex.from_documents([doc], storage_context=storage_context)

        # 🟢 จังหวะที่ 1: Embed เสร็จแล้ว! เปลี่ยนสถานะให้เว็บปลดล็อกหน้าแชททันที!
        doc_status[task_id] = {
            "status": "ready_for_chat",
            "summary": "⏳ AI กำลังสรุปเนื้อหาและสร้าง Mindmap อยู่เบื้องหลัง คุณสามารถเริ่มพิมพ์ถามตอบได้เลย...",
            "mindmap": {"nodes": [], "edges": []}
        }
        print(f"✅ [Index] {filename} สร้าง Vector เสร็จแล้ว (เริ่มแชทได้เลย)!")

        # 🟡 จังหวะที่ 2: เริ่มทำงานหนัก (Summary & Mindmap)
        llm_summarizer = Ollama(model="scb10x/typhoon2.5-qwen3-4b", request_timeout=600.0, additional_kwargs={"num_ctx": 4096})
        engine = index.as_query_engine(llm=llm_summarizer, similarity_top_k=2)

        print(f"⏳ [LLM] กำลังคิด Summary (เบื้องหลัง)...")
        summary_res = await engine.aquery("สรุปเนื้อหาสำคัญ 5 ข้อเป็นภาษาไทยแบบกระชับ")

        print(f"⏳ [LLM] กำลังแกะ Mindmap JSON (เบื้องหลัง)...")
        mindmap_res = await engine.aquery("Extract mindmap data in ONLY valid JSON format. Example: {\"nodes\": [{\"id\": \"1\", \"data\": {\"label\": \"topic\"}}], \"edges\": []}. Do not say anything else.")

        mindmap_json = {"nodes": [{"id": "1", "data": {"label": "แกะ Mindmap ไม่สำเร็จ"}}], "edges": []}
        json_match = re.search(r'\{.*\}', str(mindmap_res), re.DOTALL)
        if json_match:
            try: mindmap_json = json.loads(json_match.group(0))
            except: pass

        # 🟢 เสร็จสมบูรณ์ 100%: เปลี่ยนสถานะเป็น completed
        doc_status[task_id] = {
            "status": "completed",
            "summary": str(summary_res),
            "mindmap": mindmap_json
        }
        print(f"🎉 ประมวลผลไฟล์ {filename} เสร็จสิ้น 100%!\n")

    except Exception as e:
        doc_status[task_id] = {"status": "error", "message": str(e)}
        print(f"❌ พังระหว่างทำ Background Task: {e}")

@app.post("/upload")
async def upload_document(
    background_tasks: BackgroundTasks,
    file: UploadFile = File(...),
    session_id: str = Form(...)
):
    print(f"📥 [API] ได้รับไฟล์: {file.filename} (Session: {session_id})")
    os.makedirs("uploaded_docs", exist_ok=True)
    file_path = f"uploaded_docs/{session_id}_{file.filename}"
    with open(file_path, "wb") as buffer:
        shutil.copyfileobj(file.file, buffer)

    task_id = f"{session_id}_{file.filename}"
    doc_status[task_id] = {"status": "processing", "summary": "", "mindmap": {}}
    background_tasks.add_task(process_document_background, file_path, file.filename, session_id)
    return {"status": "success", "filename": file.filename, "session_id": session_id}

@app.get("/document/status/{session_id}/{filename}")
async def get_document_status(session_id: str, filename: str):
    return doc_status.get(f"{session_id}_{filename}", {"status": "not_found"})

class SingleChatReq(BaseModel):
    query: str
    model_name: str
    session_id: str

@app.post("/chat/single")
async def chat_single(req: SingleChatReq):
    try:
        print(f"💬 [Chat] คำถามจาก {req.session_id}: {req.query}")
        storage_context = get_session_storage(req.session_id)
        index = VectorStoreIndex.from_vector_store(vector_store=storage_context.vector_store)

        dynamic_llm = Ollama(model=req.model_name, request_timeout=600.0, additional_kwargs={"num_ctx": 4096})
        engine = index.as_query_engine(llm=dynamic_llm, similarity_top_k=2)

        print(f"🤖 [LLM] กำลังคิดคำตอบด้วยโมเดล {req.model_name}...")
        ans = await asyncio.to_thread(engine.query, req.query)
        print(f"📤 [Chat] ส่งคำตอบเรียบร้อย")
        return {"query": req.query, "answer": str(ans)}
    except Exception as e:
        print(f"❌ [Chat Error] {str(e)}")
        return {"status": "error", "message": str(e)}

nest_asyncio.apply()
NGROK_TOKEN = "3B6z594h3nMz5J6bYrJVXXF0Xuh_44KBV7bJJtmQ1heH2iCY8"
ngrok.set_auth_token(NGROK_TOKEN)
try: ngrok.kill()
except: pass

public_url = ngrok.connect(8000).public_url
print(f"🚀 API ออนไลน์ที่: {public_url}")

config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
server = uvicorn.Server(config)
await server.serve()

🚀 กำลังสตาร์ท Ollama Server...
🔓 เคลียร์ Lock File สำเร็จ
🔗 กำลังเชื่อมต่อ Database...
🚀 API ออนไลน์ที่: https://autacoidal-sectoral-helen.ngrok-free.dev


INFO:     Started server process [512]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


เอาต์พุตของการสตรีมมีการตัดเหลือเพียง 5000 บรรทัดสุดท้าย
INFO:     2405:9800:b860:976e:70b6:d56e:a552:74f:0 - "GET /document/status/%E0%B8%A3%E0%B8%B2%E0%B8%A2%E0%B8%87%E0%B8%B2%E0%B8%99%E0%B8%81%E0%B8%B2%E0%B8%A3%E0%B8%A7%E0%B8%B4%E0%B9%80%E0%B8%84%E0%B8%A3%E0%B8%B2%E0%B8%B0%E0%B8%AB%E0%B9%8C%20TOR%20%E0%B9%81%E0%B8%A5%E0%B8%B0%E0%B8%81%E0%B8%B2%E0%B8%A3%E0%B8%A7%E0%B8%B2%E0%B8%87%E0%B9%81%E0%B8%9C%E0%B8%99%E0%B8%9E%E0%B8%B1%E0%B8%92%E0%B8%99%E0%B8%B2%E0%B9%82%E0%B8%84%E0%B8%A3%E0%B8%87%E0%B8%81%E0%B8%B2%E0%B8%A3_v2.pdf/undefined HTTP/1.1" 200 OK
INFO:     2405:9800:b860:976e:70b6:d56e:a552:74f:0 - "GET /document/status/%E0%B8%A3%E0%B8%B2%E0%B8%A2%E0%B8%87%E0%B8%B2%E0%B8%99%E0%B8%81%E0%B8%B2%E0%B8%A3%E0%B8%A7%E0%B8%B4%E0%B9%80%E0%B8%84%E0%B8%A3%E0%B8%B2%E0%B8%B0%E0%B8%AB%E0%B9%8C%20TOR%20%E0%B9%81%E0%B8%A5%E0%B8%B0%E0%B8%81%E0%B8%B2%E0%B8%A3%E0%B8%A7%E0%B8%B2%E0%B8%87%E0%B9%81%E0%B8%9C%E0%B8%99%E0%B8%9E%E0%B8%B1%E0%B8%92%E0%B8%99%E0%B8%B2%E0%B9%82%E0%B8%84%E0%B8%A3%E0%B8%87%E0%B8%81%E

INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [512]
